# Module 2　量化风险和回报

**——现代投资组合理论：搭建起分析框架，从头开始**

## 导读

风险与回报同样重要，不能只看风险或回报——这是习以为常却又最容易被忽略的一点。本篇以马科维茨的均值–方差框架为主线：先把可投资资产及其回报、风险、相关性对应到 $\mu$–$\sigma$ 图上的一个点，再依次讨论两资产组合、$N$ 个风险资产与分散化，最后转入最优化问题的求解——无约束情形用梯度与 Hessian，带约束情形用拉格朗日乘子法，由此得到有效前沿、全局最小方差组合与切点组合。全篇围绕一条主线展开：**在给定的风险水平下把回报做到最大，等价于在给定的回报目标下把风险做到最小**。

本篇的统一前提如下：

- 资产数 $N \ge 2$，初始可控金额（初始资产）为 $W$；
- 在时间 $T$ 内挣得越多越好，且只可在 $N$ 个可投资资产之间买卖（做多或做空）；
- 决策一旦确定就不再改变，直到一期结束。

## 目录

- **一、可投资资产**
  - （一）资产的分类
  - （二）组合权重的定义
  - （三）风险资产的统计刻画
  - （四）椭圆分布族
- **二、两资产组合**
  - （一）两种投资思路
  - （二）无风险资产
  - （三）分析的基本前提
  - （四）无风险资产与单个风险资产
  - （五）两个风险资产
  - （六）相关系数的影响
  - （七）加入无风险资产：切点组合、CAL 与 CML
- **三、N 个风险资产与分散化**
  - （一）向量表示与组合的矩
  - （二）资产分散化
  - （三）系统性风险与跨市场操作
  - （四）含无风险资产的 N 资产组合
  - （五）效用函数与风险厌恶
  - （六）应用上的困难
- **四、最优化问题的求解**
  - （一）最优化问题的基本要素
  - （二）无约束优化：梯度与 Hessian
  - （三）无约束优化的应用：最优权重
  - （四）约束优化：拉格朗日乘子法
  - （五）有效前沿：给定回报下的最小风险
    - 1. 拉格朗日函数与一阶条件
    - 2. 标量 $A$、$B$、$C$ 与 $AC - B^2 > 0$
    - 3. 最优权重与乘子
  - （六）全局最小方差组合与 $\sigma^2(m)$
  - （七）含无风险资产：切点组合的推导
    - 1. 带单一约束的最优化
    - 2. 切点组合的权重、均值与标准差

---


## 一、可投资资产

投资的第一步，是弄清"可以买什么"。可投资产的种类繁多，风险特征也各不相同；在把它们放进组合之前，先统一分类的口径，再约定权重的算法。

### （一）资产的分类

按资产的性质，可投资产大致分为三类：

- **金融资产**：股票、外汇等；
- **真实资产**：大宗商品等；
- **无形资产**：劳动力等。

另外不要忘了现实中的限制，比如有些品种仅限国内交易等。这些限制不改变定价的逻辑，却会实实在在地压缩可选择的空间。

### （二）组合权重的定义

各个资产在组合中的权重怎么看？用数学表达，即

$$w_i = \frac{\text{Market Value of Investment in Asset } i}{\text{Total Market Value of the Portfolio}}\tag{1}$$

说白了，就是这个对应资产的市值除以总的资产市值。看起来很简单，但是咱们以债券为例：债券面值 100，市场价值却可能小于 100，我们要用的是那个**市场价值**，不能用面值。

### （三）风险资产的统计刻画

假设每一个风险资产都可以用下列三个指标完整描述。记资产为 $i$：

- **预期回报** $\mu_i$：注意是预期回报，不是实际回报；
- **风险** $\sigma_i$：即该资产回报的标准差；
- **相关性** $\rho_{ij}$：任意两个资产 $i$、$j$ 之间的相关系数。

资产组合的风险正是由这三者共同决定的，这也是均值–方差框架的全部输入。

### （四）椭圆分布族

需要说明的是，上述"均值 + 方差 + 相关系数"的描述并非万能，它成立的前提是资产的回报服从**椭圆分布族**（elliptical distribution）。

椭圆分布其实就是正态分布与 $t$ 分布——把它们的密度曲面从上往下看，等高线是一族椭圆，故得此名。它的累积分布函数（CDF），也就是我们熟悉的 $F(x)$ 概率分布（密度）函数，形式上仍然只依赖均值向量与协方差矩阵，因此用 $\mu$、$\sigma$、$\rho$ 三个量就能把分布说清楚。


## 二、两资产组合

我们的资产组合最终是为了挣钱，回报当然是在风险区间内越高越好；而当给定回报目标时，风险越小越好。

### （一）两种投资思路

马科维茨认为，理性人的投资思路本质上只有这两种：

- 给定回报目标，使风险最小；
- 给定风险水平，使回报最大。

量化就是基于这个理论，把金融问题转化为了数学问题。以回报率为 $y$ 轴、风险为 $x$ 轴，此时就是把风险当作投入、回报即为产出，任何一个资产都可以对应到这个图上的一个点上。

这一套思路就是**均值–方差最优化**（mean–variance optimization）：在均值–方差平面上比较所有可行的组合，再把"给定回报下风险最小"或"给定风险下回报最大"的那一类挑出来。

### （二）无风险资产

风险资产的回报固然高，那有没有无风险资产呢？有的兄弟，有的：正常的定期存款或债券一般就是无风险资产（RFA，Risk-Free Asset）。

无风险资产的回报率就是利率 $R$——它的风险必然为 0，且与任何资产的相关系数都是 0（无风险资产与任何资产无关）。

所以在 $\mu$–$\sigma$ 图上，RFA 就是纵轴上的一个点：风险为 0，回报为 $R$。

### （三）分析的基本前提

OK 啊，现在我们可以采用一些工具来分析了。先把分析的基本前提交代清楚，防止有扯皮：

① **总体回报率**：我们计算的都是总体回报率，包含了资本利得等全部收益；我们获得的收益也会进行再投资，以复利形式增长。

② **可以部分投资**：我们可以只投资一部分，未必必须全部投资——这是为了保证充分的分散。

③ **自由借贷**：假设投资者可以自由地按照无风险回报率借贷。

④ **做空无惩罚**：我们在做空某资产时不会有任何惩罚（现实中做空通常是要罚钱或付融券费用的）。

⑤ **无摩擦市场**：市场没有摩擦，比如没有抵押要求、没有税收等。

后三条是典型的理想化假设。现实中的借贷约束、做空成本与税负，都会让结论出现偏差。


### （四）无风险资产与单个风险资产

现在把资产放到 $\mu$–$\sigma$ 图中，从两个资产与 RFA 入手，构建投资组合。

若只有 RFA 和资产 A：设资产 A 的权重为 $w_A$，全部投资时 RFA 的权重为 $1 - w_A$，投资组合记为 $\Pi$，则

$$R_{\Pi} = w_A\, R_A + (1 - w_A)\, R\tag{2}$$

求期望：

$$\mu_{\Pi} = w_A\, \mu_A + (1 - w_A)\, R = R + w_A\left(\mu_A - R\right)\tag{3}$$

这就是"实际收益率 = 无风险收益率 + 超额收益率"嘛。

再看风险。由于 $(1 - w_A) R$ 为常数，对方差没有影响：

$$\sigma_{\Pi}^2 = \operatorname{Var}\left[w_A R_A + (1 - w_A) R\right] = w_A^2 \operatorname{Var}\left[R_A\right] = w_A^2 \sigma_A^2\tag{4}$$

故

$$\sigma_{\Pi} = w_A\, \sigma_A\tag{5}$$

改变 $w_A$ 就会改变资产的配比，组合对应的点在这条连线上移动。

而这条连线对应的方程就是

$$\mu_{\Pi} = R + \frac{\mu_A - R}{\sigma_A}\,\sigma_{\Pi} = R + k\,\sigma_{\Pi}, \qquad k = \frac{\mu_A - R}{\sigma_A}\tag{6}$$

其中 $k$ 就是大名鼎鼎的**夏普比率**，代表着承担一单位风险所获得的超额回报。

那么这条连线的右上方延伸是什么意思？本质是 $w_A > 1$，即又加大了资产 A 的权重——这就是**加杠杆**：我们通过杠杆买入更多的 A，以获取超额收益。左下方延伸呢？那也是加杠杆，只不过是做空资产 A、买入更多的 RFA，这显然是不划算的：对应的收益率甚至低于 RFA 本身的收益率，而且所谓"负的风险"在现实中很难存在。故最多只考虑右上方延伸。


### （五）两个风险资产

对于两个资产 A 和 B，由上面过程有

$$\mu_{\Pi} = w_A\, \mu_A + (1 - w_A)\,\mu_B = \mu_B + w_A\left(\mu_A - \mu_B\right)\tag{7}$$

$$\sigma_{\Pi} = \sqrt{w_A^2 \sigma_A^2 + (1 - w_A)^2 \sigma_B^2 + 2\rho_{AB}\, w_A (1 - w_A)\,\sigma_A \sigma_B}\tag{8}$$

此时化简得到的就不再是直线，而是双曲线了。左右两端的延伸同样是杠杆。

画一条竖线交双曲线于两个点，我们显然是选择上面那个点——同样的风险，收益率更高。

**定义 1（有效前沿）**　对每一个给定的风险水平，预期回报最高的那些组合所构成的集合，称为**有效前沿**。它就是"最好的投资组合"这一说法的确切含义。

先分析这个 $\sigma_{\Pi}$。由于相关系数 $\rho_{AB}$ 的范围是 $[-1, 1]$，代入化简即可得到组合标准差的上下限：

$$\left|w_A \sigma_A - (1 - w_A)\,\sigma_B\right| \;\le\; \sigma_{\Pi} \;\le\; w_A \sigma_A + (1 - w_A)\,\sigma_B\tag{9}$$

（后面一项中的 $1 - w_A$ 也可以写作 $w_B$。）也就是说，两个资产的相关性越小，组合的风险就能越低。

**定义 2（全局最小方差组合）**　开口朝右的双曲线最左端必有一个点对应着最小的风险，这一点的资产配置称为**全局最小方差组合**（The Global Minimum Variance Portfolio，GMV）。其权重为

$$w_A^{G} = \frac{\sigma_B\left(\sigma_B - \rho_{AB}\,\sigma_A\right)}{\sigma_A^2 + \sigma_B^2 - 2\rho_{AB}\,\sigma_A \sigma_B}\tag{10}$$

这个式子的得到其实就是化简求导取最小值，不多赘述。

三种特殊情形值得单独看一看。

**情形一：$\rho_{AB} = 1$。** 图形与前面"RFA + 单个风险资产"没什么区别，都是直线，只是斜率与初始值不同而已。

**情形二：$\rho_{AB} = -1$。** 此时

$$\sigma_{\Pi} = \left|w_A \sigma_A - (1 - w_A)\,\sigma_B\right| = \left|-\sigma_B + w_A\left(\sigma_A + \sigma_B\right)\right|\tag{11}$$

曲线恰好是一条折线，端点落在纵轴上。也就是说，对于完全负相关的资产，我们可以配置出完全无风险的投资组合：

$$w_A = \frac{\sigma_B}{\sigma_A + \sigma_B}, \qquad \sigma_Z = 0, \qquad \mu_Z = \mu_B + \frac{\sigma_B}{\sigma_A + \sigma_B}\left(\mu_A - \mu_B\right)\tag{12}$$

**情形三：$\rho_{AB} = 0$。** 还是双曲线。

### （六）相关系数的影响

相关系数的作用还分两种情形：

- **没有杠杆时**：若回报率相同，图形表明资产相关系数越小，组合的风险越小；
- **有杠杆时**：若回报率相同，图形表明资产相关系数越大，组合的风险越小。

原因在于：long 方与 short 方所做的操作相反，人为地改变了相关性——二者权重一正一负，乘起来之后 $\rho$ 越大，负得越多，风险就越小。

接下来说一下这两个相关资产再与 RFA 配置，直到延伸到多个资产组合。


### （七）加入无风险资产：切点组合、CAL 与 CML

先看权重结构。设风险资产内部的组合记为 Portfolio，其权重为 $w$；RFA 的权重为 $1 - w$；而 Portfolio 内部，A 占 $w_A$、B 占 $1 - w_A$，如表 1 所示。

<p align="center">表1&nbsp;&nbsp;两个风险资产与 RFA 的权重结构</p>

| 资产 | 权重 |
| --- | --- |
| A | $w \cdot w_A$ |
| B | $w \cdot (1 - w_A)$ |
| RFA | $1 - w$ |
| 合计 | $1$ |

分层地看，组合 $\Pi$ 先由"风险资产组合"与"RFA"两部分构成，前者再拆成 A 与 B；每一层的权重各自加总为 1。

我们也可以扩展，这样就会有两个集合：风险资产的集合与 RFA 的集合。此时 RFA 集合与风险资产集合的连线在 $\mu$–$\sigma$ 图中必然是直线。那么这多条直线，哪一个配置是最好的呢？

看直线斜率。因为简单看 $k = (\mu - R)/\sigma$，其中 $\mu$ 是风险资产集合的收益率，$R$ 是 RFA 的收益率，$\sigma$ 是风险资产集合的风险。总收益率越高，还要风险越小，那就是斜率越大越好。

换种说法，夏普比率越高，代表着承担一单位风险所获得的超额回报越多，这肯定是最好的。

图形上，显然是直线与双曲线相切时直线斜率最大。

**定义 3（切点组合）**　RFA 与风险资产集合的连线若恰好与风险资产的有效前沿相切，切点处的那个风险资产组合称为**切点组合**（tangency portfolio）。它是夏普比率最高的风险资产组合。

切点组合中资产 A 的权重表达式为

$$\tilde{w}_A = \frac{\sigma_B\left[\left(\mu_B - R\right)\rho_{AB}\,\sigma_A - \left(\mu_A - R\right)\sigma_B\right]}{-\left(\mu_A - R\right)\sigma_B^2 - \left(\mu_B - R\right)\sigma_A^2 + \left(\mu_A + \mu_B - 2R\right)\rho_{AB}\,\sigma_A \sigma_B}\tag{13}$$

其中 $\tilde{w}_A$ 表示切点组合内部资产 A 所占的权重。

这条切线，对于个人投资者就是大名鼎鼎的 **CAL 线**（Capital Allocation Line），对于市场（千千万万个个人投资者的汇总）就是 **CML 线**（Capital Market Line）。

> 注：只有风险资产时，对应的是有效前沿；RFA 集合与风险资产集合合并，才对应 CAL 或 CML。

当然哈，这里既然有 $\rho$，那也会有相关系数取值上的讨论，同时当配置无风险组合时也能与 Module 1 的内容相呼应。不多赘述，方法显而易见。


## 三、N 个风险资产与分散化

终于扩展到 $N$ 个风险资产了。因为这种需要配置的资产很多，我们用向量（矩阵）表示，方便计算。

### （一）向量表示与组合的矩

首先回到大前提：资产配置权重加和等于 1，即

$$\mathbf{w}^{T}\mathbf{1}_N = 1\tag{14}$$

其中 $\mathbf{w} = (w_1, w_2, \ldots, w_N)^{T}$ 是一个 $N$ 维列向量，$\mathbf{w}^{T}$ 就是它的转置；$\mathbf{1}_N$ 是 $N$ 维单位列向量。

再设

$$\boldsymbol{\mu} = (\mu_1, \mu_2, \ldots, \mu_N)^{T}, \qquad \boldsymbol{\sigma} = (\sigma_1, \sigma_2, \ldots, \sigma_N)^{T}$$

其中 $\mu_i$、$\sigma_i$ 分别是资产 $i$ 的预期回报与风险。于是组合的均值与标准差为

$$\mu_{\Pi} = \mathbf{w}^{T}\boldsymbol{\mu}, \qquad \sigma_{\Pi} = \sqrt{\mathbf{w}^{T}\boldsymbol{\Sigma}\mathbf{w}}\tag{15}$$

其中 $\boldsymbol{\Sigma}$ 为协方差矩阵：

$$\boldsymbol{\Sigma} = \begin{pmatrix}
\sigma_1^2 & \rho_{12}\sigma_1\sigma_2 & \cdots & \rho_{1N}\sigma_1\sigma_N \\
\rho_{21}\sigma_2\sigma_1 & \sigma_2^2 & \cdots & \rho_{2N}\sigma_2\sigma_N \\
\vdots & \vdots & \ddots & \vdots \\
\rho_{N1}\sigma_N\sigma_1 & \rho_{N2}\sigma_N\sigma_2 & \cdots & \sigma_N^2
\end{pmatrix}\tag{16}$$

记住协方差矩阵长这个样子。注意它其实是个对称矩阵，计算什么的是很方便的。

对于 $N$ 个风险资产的组合，我们会画出一个面而非两个风险资产的线，更能凸显有效前沿的特点。


### （二）资产分散化

假设市场上所有的资产都是同质化的：预期回报都是 $\mu$，风险都是 $\sigma$，任意两个资产之间的相关系数都是 $\rho$，权重雨露均沾，都是 $w = 1/N$。

显然，组合的回报仍然为 $\mu$，但风险是

$$\sigma_{\Pi}^2 = \left(\rho + \frac{1 - \rho}{N}\right)\sigma^2\tag{17}$$

当 $N \to \infty$ 时，

$$\sigma_{\Pi}^2 \to \rho\,\sigma^2\tag{18}$$

这就是分散资产可以使风险变小的数学证明。

### （三）系统性风险与跨市场操作

现实生活中，配置 30~50 个资产就基本达到了极限。事实上，风险分散会不断逼近某个值，这个风险就是众所周知的**系统性风险**，也就是不可被分散的风险。

但是！由于不同市场的 $\beta$ 不同，也就是系统性风险的大小不同，我们可以通过**跨市场操作**来降低系统性风险。当然啦，全球性风险依然是不可以被分散的。

### （四）含无风险资产的 N 资产组合

配上 RFA 集合，我们设

$$w_0 = 1 - \mathbf{w}^{T}\mathbf{1}_N\tag{19}$$

通过简单的化简就有

$$\mu_{\Pi} = R + \mathbf{w}^{T}\left(\boldsymbol{\mu} - \mathbf{1}_N R\right), \qquad \sigma_{\Pi} = \sqrt{\mathbf{w}^{T}\boldsymbol{\Sigma}\mathbf{w}}\tag{20}$$

（这里只有 $R$ 不是粗体，它只是个标量，不是向量或矩阵。）

### （五）效用函数与风险厌恶

回归到马科维茨的理论，我们就两种投资方式。现实中由于风险有更多方法可以控制，而所谓的"追求更高回报"不可控，故更常见的是固定回报率使之风险最小。

假设没有给定的风险与回报率，我们可以将二者结合起来，构造拉格朗日函数，生成一个效用函数：

$$u = \max_{\mathbf{w}}\left\{ E\left[R_p(w_1, \ldots, w_N)\right] - \frac{\lambda}{2}\,\sigma_p^2(w_1, \ldots, w_N) \right\}, \qquad \text{s.t. } \sum_{i=1}^{N} w_i = 1\tag{21}$$

其中 $\lambda$ 是风险厌恶程度系数。现实中这是比较难确定的，一般都是问卷调查（hhh）。

### （六）应用上的困难

应用上，这个计算简直是太麻烦了。原理很简单，我们只能用一些算法或模型来简化优化；但是怎么判断问题是否为线性的、用什么模型或因子来解决问题，都很难弄。

虽然相关系数相对较稳定，但是风险的测量与回报率的测算很好地弥补了这一点。实在不行，就用其他的分析方法吧。


## 四、最优化问题的求解

前面所有的组合问题，最后都落到同一句话上：在一定的条件下，求某个目标的最大值或最小值。本章先把最优化问题的语言讲清楚，再分别处理无约束与带约束两种情形，最后回到有效前沿与切点组合。

### （一）最优化问题的基本要素

简单介绍：最优化就是在一定的条件下求出最小值或最大值。广泛地说，我们要学会用这种理论培养一种合适的组合意识，当然在这里我们就是应用于资产组合。

要素如下：

1. **目标函数**：设 $f$ 为目标函数，则 $\min f(x_1, x_2, \ldots, x_n)$ 中的 $f$ 就是关于决策变量的函数。
2. **约束条件**：$g_k(x_1, x_2, \ldots, x_n)$，$k = 1, 2, 3, \ldots$，要么大于等于、要么小于等于、要么等于某个确定的约束限制 $b_k$。
3. **一些简单的技巧**，方便求解与编程：

$$\max f(x) = -\min\left(-f(x)\right), \qquad \min f(x) = -\max\left(-f(x)\right)$$

$$\max\left(a + b\,f(x)\right) = a + b\,\max f(x), \qquad b > 0\tag{22}$$

OK 啊，举个例子。众所周知，BS 模型那个大长串公式可以表示为

$$V(S, X, R, \sigma, t) = P$$

此时这个 $V$ 就是目标函数 $f$：我们已知目标值 $P$，反求里面的风险 $\sigma$ 等决策变量。


### （二）无约束优化：梯度与 Hessian

我们把决策变量化作向量的形式：$\mathbf{x} = (x_1, x_2, \ldots, x_n)^{T}$（这是个列向量）。

我们找最小值，不就是求一阶导、找到一阶导为 0 的点，再求二阶导判断是最大值还是最小值嘛。那么，利用梯度（一个倒三角符号，写作 $\nabla$）：

$$\nabla f(\mathbf{x}) = Df = \left(\frac{\partial f}{\partial x_1}, \frac{\partial f}{\partial x_2}, \ldots, \frac{\partial f}{\partial x_n}\right)^{T}\tag{23}$$

（依旧是个列向量。）然后求二阶导，得出 Hessian 矩阵：

$$H f(\mathbf{x}) = D^{2}f = \begin{pmatrix}
\dfrac{\partial^2 f}{\partial x_1^2} & \dfrac{\partial^2 f}{\partial x_2 \partial x_1} & \cdots & \dfrac{\partial^2 f}{\partial x_n \partial x_1} \\
\dfrac{\partial^2 f}{\partial x_1 \partial x_2} & \dfrac{\partial^2 f}{\partial x_2^2} & \cdots & \vdots \\
\vdots & \vdots & \ddots & \vdots \\
\dfrac{\partial^2 f}{\partial x_1 \partial x_n} & \cdots & \cdots & \dfrac{\partial^2 f}{\partial x_n^2}
\end{pmatrix}\tag{24}$$

长成这样，这显然也是个对称矩阵。

假设对应最值的横坐标为 $\mathbf{x}^{*}$，那么就有

- **极小值**：$\nabla f(\mathbf{x}^{*}) = 0$，$H f(\mathbf{x}^{*}) > 0$；
- **极大值**：$\nabla f(\mathbf{x}^{*}) = 0$，$H f(\mathbf{x}^{*}) < 0$。

万一没有极值，那就看可不可以先用约束条件约束一下再求解，否则就判定为无极值。最后还要看边界，找出边界上的最值。这是简单的数学思路。


### （三）无约束优化的应用：最优权重

我们做一次简单应用。依旧假设有 $n$ 个资产，权重 $w_i$ 等等初始设置都是一样的，也有

$$\mu_{\Pi} = \mathbf{w}^{T}\boldsymbol{\mu}, \qquad \sigma_{\Pi} = \sqrt{\mathbf{w}^{T}\boldsymbol{\Sigma}\mathbf{w}}\tag{25}$$

其中 $\boldsymbol{\Sigma}$ 为协方差矩阵。为了方便计算，我们把它拆成相关系数矩阵 $\mathbf{R}$：

$$\mathbf{R} = \begin{pmatrix}
1 & \rho_{12} & \cdots & \rho_{1n} \\
\rho_{21} & 1 & \cdots & \rho_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
\rho_{n1} & \cdots & \cdots & 1
\end{pmatrix}\tag{26}$$

和标准差对角矩阵 $\mathbf{S}$：

$$\mathbf{S} = D(\boldsymbol{\sigma}) = \begin{pmatrix}
\sigma_1 & 0 & \cdots & 0 \\
0 & \sigma_2 & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & \cdots & \cdots & \sigma_n
\end{pmatrix}\tag{27}$$

由此推得

$$\boldsymbol{\Sigma} = \mathbf{S}^{T}\mathbf{R}\mathbf{S} = \mathbf{S}\mathbf{R}\mathbf{S}\tag{28}$$

（因为对角矩阵的转置就是它本身。）这样的计算方式是风险度量体系的基本工具。

其实 RFA 与任何资产的相关系数都等于 0，所以只考虑风险资产的配比，RFA 的权重为 $w_0 = 1 - \mathbf{w}^{T}\mathbf{1}_N$。于是

$$\mu_{\Pi} = R + \mathbf{w}^{T}\left(\boldsymbol{\mu} - \mathbf{1}_N R\right), \qquad \sigma_{\Pi} = \sqrt{\mathbf{w}^{T}\boldsymbol{\Sigma}\mathbf{w}}$$

接下来用

$$\max_{\mathbf{w}}\; \mu_{\Pi} - \frac{\lambda}{2}\,\sigma_{\Pi}^2\tag{29}$$

来计算，其中 $\lambda$ 为风险厌恶系数，$\mathbf{w}$ 是风险资产权重。把上面的 $\mu_{\Pi}$、$\sigma_{\Pi}$ 代入进去有

$$\max_{\mathbf{w}}\; V(\mathbf{w}) = \left[R + \mathbf{w}^{T}\left(\boldsymbol{\mu} - \mathbf{1}_N R\right)\right] - \frac{\lambda}{2}\,\mathbf{w}^{T}\boldsymbol{\Sigma}\mathbf{w}\tag{30}$$

求一阶导，就是

$$\nabla V(\mathbf{w}^{*}) = \frac{\partial V}{\partial \mathbf{w}}(\mathbf{w}^{*}) = \left(\boldsymbol{\mu} - \mathbf{1}_N R\right) - \lambda\,\boldsymbol{\Sigma}\mathbf{w}^{*} = 0\tag{31}$$

（这里用到了矩阵求导法则，见本节末的注。）整理一下就是

$$\mathbf{w}^{*} = \frac{1}{\lambda}\,\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - \mathbf{1}_N R\right)\tag{32}$$

求二阶导：

$$H V(\mathbf{w}^{*}) = -\lambda\,\boldsymbol{\Sigma} < 0\tag{33}$$

这就是求出来了最大值。

> 注：这里用到的两条矩阵求导法则为 $\frac{\partial(\mathbf{a}^{T}\mathbf{w})}{\partial \mathbf{w}} = \mathbf{a}$ 与 $\frac{\partial(\mathbf{w}^{T}\boldsymbol{\Sigma}\mathbf{w})}{\partial \mathbf{w}} = \left(\boldsymbol{\Sigma} + \boldsymbol{\Sigma}^{T}\right)\mathbf{w} = 2\boldsymbol{\Sigma}\mathbf{w}$（$\boldsymbol{\Sigma}$ 对称）。负定性判据则沿用了标量情形的结论：$-\lambda\boldsymbol{\Sigma} < 0$ 意味着 $V$ 在 $\mathbf{w}^{*}$ 处取极大值。


### （四）约束优化：拉格朗日乘子法

带约束的不能用微积分求解，数学证明略。我们可以用拉格朗日方法解决问题。

**第一步：构造拉格朗日函数。**

$$L(\mathbf{x}, \boldsymbol{\lambda}) = f(x_1, x_2, \ldots, x_n) + \sum_{j=1}^{m}\lambda_j\left(g_j(\mathbf{x}) - b_j\right)\tag{34}$$

此处的 $\lambda_j$ 为拉格朗日乘子，每一个都对应一个优化条件。（顺带说一下：这里的 $\mathbf{x}$ 与 $\boldsymbol{\lambda}$ 都是向量，只是因为写成了加和的形式，此处不再统一加粗。）这样我们就把 $n$ 个带约束的决策变量转化为了 $n + m$ 个不带约束的决策变量。此时就会求

$$\min_{\mathbf{x}, \boldsymbol{\lambda}} L(\mathbf{x}, \boldsymbol{\lambda})\tag{35}$$

**第二步：求一阶导。**

$$\frac{\partial L}{\partial x_i}(\mathbf{x}) = \frac{\partial f}{\partial x_i}(\mathbf{x}) + \sum_{j=1}^{m}\lambda_j\,\frac{\partial g_j}{\partial x_i}(\mathbf{x}) = 0, \qquad i = 1, \ldots, n\tag{36}$$

$$\frac{\partial L}{\partial \lambda_j}(\mathbf{x}) = g_j(\mathbf{x}) - b_j = 0, \qquad j = 1, \ldots, m\tag{37}$$

**第三步：求二阶导**，再判断所得驻点是极大值还是极小值。

一阶条件与二阶条件合起来，就是带约束优化问题的标准解法。


### （五）有效前沿：给定回报下的最小风险

#### 1. 拉格朗日函数与一阶条件

举个例子。拿经典的有效前沿面图来看，我们要计算的是：当收益为定值时，最小风险是多少。这就是一个带约束的优化问题。

我们先表示一下各个参数，再代入这套方法框架求解。目标是

$$\min_{\mathbf{w}}\; \frac{1}{2}\sigma_{\Pi}^2 = \frac{1}{2}\,\mathbf{w}^{T}\boldsymbol{\Sigma}\mathbf{w}\tag{38}$$

（乘上 $1/2$ 后会方便计算。）约束条件有两个：

1. **回报率**：$\mu_{\Pi} = \mathbf{w}^{T}\boldsymbol{\mu} = m$，要等于我们设定的目标值 $m$；
2. **预算约束**：我们这里的有效前沿图未考虑 RFA，故 $\mathbf{w}^{T}\mathbf{1}_N = 1$。

于是目标函数与这两个约束一起，代入到拉格朗日函数里有

$$L(\mathbf{w}, \lambda, \gamma) = \frac{1}{2}\,\mathbf{w}^{T}\boldsymbol{\Sigma}\mathbf{w} + \lambda\left(m - \mathbf{w}^{T}\boldsymbol{\mu}\right) + \gamma\left(1 - \mathbf{w}^{T}\mathbf{1}_N\right)\tag{39}$$

对 $\mathbf{w}$ 求一阶导：

$$\frac{\partial L}{\partial \mathbf{w}}\left(\mathbf{w}, \lambda, \gamma\right) = \boldsymbol{\Sigma}\mathbf{w} - \lambda\,\boldsymbol{\mu} - \gamma\,\mathbf{1}_N = 0\tag{40}$$

整理一下得到

$$\mathbf{w}^{*} = \boldsymbol{\Sigma}^{-1}\left(\lambda\,\boldsymbol{\mu} + \gamma\,\mathbf{1}_N\right)\tag{41}$$

然后找二阶导：发现它一定等于 $\boldsymbol{\Sigma}$，而 $\boldsymbol{\Sigma}$ 是个正定矩阵，必然大于 0，所以得到了函数的最小值。

接下来把 $\mathbf{w}^{*}$ 代回约束方程，解出乘子 $\lambda$、$\gamma$。


代入回报率约束，得

$$\boldsymbol{\mu}^{T}\boldsymbol{\Sigma}^{-1}\left(\lambda\boldsymbol{\mu} + \gamma\mathbf{1}_N\right) = \lambda\,\boldsymbol{\mu}^{T}\boldsymbol{\Sigma}^{-1}\boldsymbol{\mu} + \gamma\,\boldsymbol{\mu}^{T}\boldsymbol{\Sigma}^{-1}\mathbf{1}_N = m\tag{42}$$

再代入预算约束，得

$$\mathbf{1}_N^{T}\boldsymbol{\Sigma}^{-1}\left(\lambda\boldsymbol{\mu} + \gamma\mathbf{1}_N\right) = \lambda\,\mathbf{1}_N^{T}\boldsymbol{\Sigma}^{-1}\boldsymbol{\mu} + \gamma\,\mathbf{1}_N^{T}\boldsymbol{\Sigma}^{-1}\mathbf{1}_N = 1\tag{43}$$

#### 2. 标量 $A$、$B$、$C$ 与 $AC - B^2 > 0$

为了方便起见，我们定义如下三个标量：

$$A = \mathbf{1}_N^{T}\boldsymbol{\Sigma}^{-1}\mathbf{1}_N, \qquad B = \boldsymbol{\mu}^{T}\boldsymbol{\Sigma}^{-1}\mathbf{1}_N = \mathbf{1}_N^{T}\boldsymbol{\Sigma}^{-1}\boldsymbol{\mu}, \qquad C = \boldsymbol{\mu}^{T}\boldsymbol{\Sigma}^{-1}\boldsymbol{\mu}\tag{44}$$

于是乘子所满足的方程组化为

$$C\lambda + B\gamma = m, \qquad B\lambda + A\gamma = 1\tag{45}$$

解得

$$\lambda = \frac{Am - B}{AC - B^2}, \qquad \gamma = \frac{C - Bm}{AC - B^2}\tag{46}$$

这里要单独解释一下 $AC - B^2 > 0$ 从何而来。

**理由一（Cauchy–Schwarz）。** 令 $\mathbf{u} = \boldsymbol{\Sigma}^{-1/2}\mathbf{1}_N$、$\mathbf{v} = \boldsymbol{\Sigma}^{-1/2}\boldsymbol{\mu}$，则

$$A = \|\mathbf{u}\|^2, \qquad C = \|\mathbf{v}\|^2, \qquad B = \mathbf{u}^{T}\mathbf{v}$$

由 Cauchy–Schwarz 不等式，$\left(\mathbf{u}^{T}\mathbf{v}\right)^2 \le \|\mathbf{u}\|^2\|\mathbf{v}\|^2$，即

$$B^2 \le AC\tag{47}$$

等号成立当且仅当 $\mathbf{u}$ 与 $\mathbf{v}$ 共线，也就是 $\boldsymbol{\mu}$ 与 $\mathbf{1}_N$ 成正比、所有资产的预期回报相同。而风险资产之间的预期回报一般并不相同，故取严格不等号：$AC - B^2 > 0$。

**理由二（正定性）。** 也可以把 $A$、$B$、$C$ 看作矩阵

$$\begin{pmatrix} A & B \\ B & C \end{pmatrix} = \begin{pmatrix} \mathbf{1}_N^{T} \\ \boldsymbol{\mu}^{T} \end{pmatrix}\boldsymbol{\Sigma}^{-1}\begin{pmatrix} \mathbf{1}_N & \boldsymbol{\mu} \end{pmatrix}$$

的四个元素。由于 $\boldsymbol{\Sigma}^{-1}$ 正定，而 $\mathbf{1}_N$ 与 $\boldsymbol{\mu}$ 线性无关，该矩阵正定，其行列式 $AC - B^2 > 0$。

这个不等式有两处用处：其一，它保证式(45)的系数矩阵可逆，乘子 $\lambda$、$\gamma$ 唯一确定；其二，后文全局最小方差组合的方差 $\sigma_g^2 = 1/A$ 与最优权重都以 $AC - B^2$ 为分母，必须不为零。

#### 3. 最优权重与乘子

最终得到

$$\mathbf{w}^{*} = \frac{1}{AC - B^2}\,\boldsymbol{\Sigma}^{-1}\left[\left(A\boldsymbol{\mu} - B\mathbf{1}_N\right)m + \left(C\mathbf{1}_N - B\boldsymbol{\mu}\right)\right]\tag{48}$$

> 注：注意精细度不高时可以用数值解，更快——使用迭代法计算，小数点后 6 位以内都行。


### （六）全局最小方差组合与 $\sigma^2(m)$

最后解一下全局最小方差组合对应的权重 $\mathbf{w}$。此时就是

$$\min_{\mathbf{w}}\; \frac{1}{2}\,\mathbf{w}^{T}\boldsymbol{\Sigma}\mathbf{w}, \qquad \text{s.t. } \mathbf{w}^{T}\mathbf{1}_N = 1\tag{49}$$

除了正常求解的方法，我们可以假设有无穷个 $m$，一定会有唯一的一个 $m$ 对应着我们的目标。所以直接就去找那个唯一的 $m$ 的值就好了。

我们可以采用构建函数 $\sigma_{\Pi}^2(m)$ 的方法，使 $\sigma_{\Pi}^2(m)$ 为目标函数、$m$ 为决策变量。先把式(46)的乘子写回 $\mathbf{w}^{*}$：

$$\mathbf{w}^{*}(m) = \boldsymbol{\Sigma}^{-1}\left(\lambda(m)\,\boldsymbol{\mu} + \gamma(m)\,\mathbf{1}_N\right), \qquad \lambda(m) = \frac{Am - B}{AC - B^2}, \quad \gamma(m) = \frac{C - Bm}{AC - B^2}\tag{50}$$

（记住 $A$、$B$、$C$ 只是标量参数！）

再把它代入 $\sigma_{\Pi}^2 = \mathbf{w}^{T}\boldsymbol{\Sigma}\mathbf{w}$。记 $D = AC - B^2$，则

$$\sigma_{\Pi}^2(m) = \left(\lambda\boldsymbol{\mu} + \gamma\mathbf{1}_N\right)^{T}\boldsymbol{\Sigma}^{-1}\left(\lambda\boldsymbol{\mu} + \gamma\mathbf{1}_N\right) = \lambda^2 C + 2\lambda\gamma B + \gamma^2 A$$

代入 $\lambda$、$\gamma$ 并通分：

$$\sigma_{\Pi}^2(m) = \frac{C\left(Am - B\right)^2 + 2B\left(Am - B\right)\left(C - Bm\right) + A\left(C - Bm\right)^2}{D^2}$$

分子按 $m$ 的幂次合并：$m^2$ 项为 $A^2C - 2AB^2 + AB^2 = A\,D$；$m$ 项为 $-2ABC + 2ABC - 2ABC + 2B^3 = -2B\,D$；常数项为 $CB^2 - 2B^2C + AC^2 = C\,D$。故

$$\sigma_{\Pi}^2(m) = \frac{Am^2 - 2Bm + C}{AC - B^2}\tag{51}$$

这就是构建好的函数：它不过是关于 $m$ 的二次函数再除以正常数 $AC - B^2$，图像是一条开口朝上的抛物线。

对 $m$ 求一阶导并令其为零：

$$\frac{d\sigma_{\Pi}^2(m)}{dm} = \frac{2Am - 2B}{AC - B^2} = 0 \quad \Longleftrightarrow \quad m_g = \frac{B}{A}\tag{52}$$

二阶导 $\dfrac{2A}{AC - B^2} > 0$（因 $A > 0$、$AC - B^2 > 0$），故 $m_g$ 确实是最小值点，$m_g$ 就是全局最小方差组合的回报。

拿到 $m_g$ 就好办了。代回式(50)：

$$\lambda(m_g) = \frac{A \cdot \frac{B}{A} - B}{AC - B^2} = 0, \qquad \gamma(m_g) = \frac{C - B \cdot \frac{B}{A}}{AC - B^2} = \frac{AC - B^2}{A\left(AC - B^2\right)} = \frac{1}{A}$$

于是权向量为

$$\mathbf{w}_g = \frac{\boldsymbol{\Sigma}^{-1}\mathbf{1}_N}{A} = \frac{\boldsymbol{\Sigma}^{-1}\mathbf{1}_N}{\mathbf{1}_N^{T}\boldsymbol{\Sigma}^{-1}\mathbf{1}_N}\tag{53}$$

以及方差

$$\sigma_g^2 = \frac{1}{A}\tag{54}$$

顺带验证一下：$\sigma_g^2 = \mathbf{w}_g^{T}\boldsymbol{\Sigma}\mathbf{w}_g = \dfrac{1}{A^2}\,\mathbf{1}_N^{T}\boldsymbol{\Sigma}^{-1}\mathbf{1}_N = \dfrac{A}{A^2} = \dfrac{1}{A}$，与式(54)一致。


### （七）含无风险资产：切点组合的推导

前面的约束优化都没有考虑 RFA。把 RFA 加进来之后，问题就变成：在给定目标回报 $m$ 的前提下，如何配置风险资产与无风险资产，使风险最小。

（下面公式都按前面的约定：无风险利率记作 $R$ 而不是 $r$，转置一律记作 $^{T}$。）

#### 1. 带单一约束的最优化

$$\min_{\mathbf{w}}\; \frac{1}{2}\,\mathbf{w}^{T}\boldsymbol{\Sigma}\mathbf{w}, \qquad \text{s.t. } R + \mathbf{w}^{T}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right) = m\tag{55}$$

拉格朗日函数为

$$L(\mathbf{w}, \lambda) = \frac{1}{2}\,\mathbf{w}^{T}\boldsymbol{\Sigma}\mathbf{w} + \lambda\left(m - R - \mathbf{w}^{T}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)\right)$$

对 $\mathbf{w}$ 求一阶条件：

$$\frac{\partial L}{\partial \mathbf{w}} = \boldsymbol{\Sigma}\mathbf{w} - \lambda\left(\boldsymbol{\mu} - R\mathbf{1}_N\right) = 0\tag{56}$$

再检查二阶条件：目标函数的 Hessian 仍然是协方差矩阵，而它是正定的，因此得到最优权重

$$\mathbf{w}^{*} = \lambda\,\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)\tag{57}$$

（这里同样是把式子转置、再在两边同乘 $\boldsymbol{\Sigma}^{-1}$ 得到的。）

把 $\mathbf{w}^{*}$ 代回约束：

$$R + \lambda\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)^{T}\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right) = m\tag{58}$$

解出乘子：

$$\lambda = \frac{m - R}{\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)^{T}\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)}\tag{59}$$

再代回式(57)，得到 $\mathbf{w}^{*}$ 的最终表达式：

$$\mathbf{w}^{*} = \frac{(m - R)\,\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)}{\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)^{T}\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)}\tag{60}$$


#### 2. 切点组合的权重、均值与标准差

**切点组合**（tangency portfolio）是全部投资于风险资产的那个组合。它的权重记为 $\mathbf{w}_t$：

$$\mathbf{w}_t = \frac{\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)}{B - AR}\tag{61}$$

这个组合的均值与标准差为

$$m_t = \mathbf{w}_t^{T}\boldsymbol{\mu} = \frac{C - BR}{B - AR}\tag{62}$$

$$\sigma_t = \sqrt{\mathbf{w}_t^{T}\boldsymbol{\Sigma}\mathbf{w}_t} = \sqrt{\frac{C - 2RB + R^2 A}{\left(B - AR\right)^2}}\tag{63}$$

**这些是怎么得到的？** 关键在于切点组合很特殊，它同时满足两件事。

其一，它在"风险资产 + 无风险资产"的新有效前沿（即 CML）上，故其权重满足式(60)：

$$\mathbf{w}_t = \frac{(m_t - R)\,\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)}{\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)^{T}\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)}$$

其二，它又在"纯风险资产"的旧有效前沿（双曲线）上，故其权重也满足式(41)：

$$\mathbf{w}_t = \boldsymbol{\Sigma}^{-1}\left(\lambda\,\boldsymbol{\mu} + \gamma\,\mathbf{1}_N\right)$$

其中 $\lambda$、$\gamma$ 由式(46)给出。

又因为切点组合全部投资于风险资产，它必须满足预算约束：

$$\mathbf{1}_N^{T}\mathbf{w}_t = 1\tag{64}$$

把式(60)代入，得

$$\mathbf{1}_N^{T}\,\frac{(m_t - R)\,\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)}{\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)^{T}\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)} = 1$$

先化分母。由 $A$、$B$、$C$ 的定义展开：

$$\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)^{T}\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right) = C - 2RB + R^2 A\tag{65}$$

于是条件化为

$$\frac{(m_t - R)\,\mathbf{1}_N^{T}\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)}{C - 2RB + R^2 A} = 1\tag{66}$$

再化分子：$\mathbf{1}_N^{T}\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right) = B - AR$。两边同乘 $C - 2RB + R^2 A$，得

$$\left(m_t - R\right)\left(B - AR\right) = C - 2RB + R^2 A\tag{67}$$

把左端展开，并把 $R\left(B - AR\right) = RB - R^2 A$ 移到右端：

$$m_t\left(B - AR\right) = C - 2RB + R^2 A + RB - R^2 A = C - RB\tag{68}$$

这就得到了切点组合的回报：

$$m_t = \frac{C - RB}{B - AR}\tag{69}$$

再把它代回式(60)。分母由式(65)给出，分子中的 $m_t - R$ 由式(67)给出，于是

$$\mathbf{w}_t = \frac{(m_t - R)\,\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)}{C - 2RB + R^2 A} = \frac{C - 2RB + R^2 A}{B - AR}\cdot\frac{\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)}{C - 2RB + R^2 A} = \frac{\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)}{B - AR}$$

即式(61)。至于标准差，把式(61)代入 $\sigma_t^2 = \mathbf{w}_t^{T}\boldsymbol{\Sigma}\mathbf{w}_t$：

$$\sigma_t^2 = \frac{\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)^{T}\boldsymbol{\Sigma}^{-1}\left(\boldsymbol{\mu} - R\mathbf{1}_N\right)}{\left(B - AR\right)^2} = \frac{C - 2RB + R^2 A}{\left(B - AR\right)^2}$$

开方即得式(63)。

至此，含无风险资产的最优配置问题全部解完。切点组合的意义在于：**所有投资者（无论风险偏好如何）持有相同的风险资产组合，只是在切点组合与无风险资产之间调整资金比例**——这正是"切线斜率（夏普比率）最大"的直接结果。

我们终于结束了优化求解的全部。


## 五、Black–Litterman 模型